# Day 078 — Exercise 2: describe_media

**What you'll build:** The image description pipeline — accepts PIL Image or file path, returns a vision LLM description.

**Why it matters:** `describe_media` is the vision pipeline of the studio. Supporting both input forms means it works in programmatic pipelines (PIL Images from camera/generation) and batch file processing.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(w=100, h=100, color=(100, 150, 200)):
    return _PILImage.new('RGB', (w, h), color=color)

_mock_describe_fn   = lambda img, q: 'A test image with a solid color background.'
_mock_transcribe_fn = lambda src: {'text': 'Hello world.', 'segments': [
    {'start': 0.0, 'end': 1.0, 'text': 'Hello world.'}]}
_mock_tts_fn        = lambda text, voice, rate, pitch: b'AUDIO:' + text[:8].encode()
from pathlib import Path

MEDIA_EXTENSIONS = {
    'image': {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff'},
    'audio': {'.mp3', '.wav', '.ogg', '.flac', '.m4a', '.aac'},
    'video': {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'},
}

def detect_media_type(path):
    ext = Path(path).suffix.lower()
    for media_type, extensions in MEDIA_EXTENSIONS.items():
        if ext in extensions:
            return media_type
    return 'unknown'


## Task

`describe_media(source, describe_fn=None) -> str`

1. `from PIL import Image`
2. `if isinstance(source, (str, Path)): image = Image.open(source)` else: `image = source`
3. `prompt = 'Describe this image in detail, including all visible content.'`
4. If `describe_fn`: `return describe_fn(image, prompt)`
5. Else: BytesIO + `image.save(buf, format='PNG')` + base64 + `ollama.chat(model='llava', ...)` + return `resp['message']['content']`

## Your Implementation

In [ ]:
import io, base64
from pathlib import Path

def describe_media(source, describe_fn=None):
    """Describe an image (PIL Image or path) using a vision LLM."""
    raise NotImplementedError


In [ ]:
import io, base64

def describe_media(source, describe_fn=None):
    from PIL import Image
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    else:
        image = source
    prompt = 'Describe this image in detail, including all visible content.'
    if describe_fn is not None:
        return describe_fn(image, prompt)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}],
    )
    return resp['message']['content']


## Automated checks

In [ ]:

score, total = 0, 4
try:
    from PIL import Image as PILImage
    import tempfile, os

    img = _make_mock_image()
    result = describe_media(img, describe_fn=_mock_describe_fn)
    assert isinstance(result, str) and len(result) > 0
    score += 1; print("✅ describe_media returns str for PIL Image input")

    received = {}
    def _dfn(i, q): received.update(img=i, q=q); return 'DESC'
    describe_media(img, describe_fn=_dfn)
    assert isinstance(received.get('img'), PILImage.Image)
    assert isinstance(received.get('q'), str) and len(received['q']) > 10
    score += 1; print("✅ describe_fn receives (PIL Image, prompt_str)")

    # File path input
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        img.save(f, format='PNG')
        tmp = f.name
    try:
        path_result = describe_media(tmp, describe_fn=_mock_describe_fn)
        assert isinstance(path_result, str)
        score += 1; print("✅ describe_media accepts file path")
    finally:
        os.unlink(tmp)

    # Both forms produce same result with same mock
    img2 = _make_mock_image()
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        img2.save(f, format='PNG')
        tmp2 = f.name
    try:
        r1 = describe_media(img2, describe_fn=_mock_describe_fn)
        r2 = describe_media(tmp2, describe_fn=_mock_describe_fn)
        assert r1 == r2
        score += 1; print("✅ PIL Image and file path produce same mock result")
    finally:
        os.unlink(tmp2)

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import io, base64

def describe_media(source, describe_fn=None):
    from PIL import Image
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    else:
        image = source
    prompt = 'Describe this image in detail, including all visible content.'
    if describe_fn is not None:
        return describe_fn(image, prompt)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}],
    )
    return resp['message']['content']
```

**Why does `describe_fn` receive PIL Image, not base64?** The conversion to base64 is an implementation detail of the Ollama call. Mocks are simpler and more versatile when they receive a PIL Image — they can inspect size, mode, and pixels if needed.

</details>